In [20]:
##Garantir o endereços de toda estrutura GOLD
from pathlib import Path

BASE = Path("../data/gold")
for path in BASE.rglob("*.parquet"):
    print(path)

..\data\gold\facts\fato_alfabetizacao_brasil\fato_alfabetizacao_brasil.parquet
..\data\gold\facts\fato_alfabetizacao_municipio\fato_alfabetizacao_municipio.parquet
..\data\gold\facts\fato_alfabetizacao_uf\fato_alfabetizacao_uf.parquet
..\data\gold\dimensions\dim_municipio\dim_municipio.parquet
..\data\gold\dimensions\dim_rede\dim_rede.parquet
..\data\gold\dimensions\dim_tempo\dim_tempo.parquet
..\data\gold\dimensions\dim_uf\dim_uf.parquet


## Explorando as dimensões

### Dimensão Municipio

In [38]:
import pandas as pd

df_municipio = pd.read_parquet("../data/gold/dimensions/dim_municipio/dim_municipio.parquet")

df_municipio.shape
#df_municipio.head(10)

(5549, 3)

In [27]:
df_municipio.head(10)

,id_municipio,Municipio,UF
0,5200050,Abadia de Goiás,GO
1,3100104,Abadia dos Dourados,MG
2,5200100,Abadiânia,GO
3,1500107,Abaetetuba,PA
4,3100203,Abaeté,MG
5,2300101,Abaiara,CE
6,2900207,Abaré,BA
7,4100103,Abatiá,PR
8,2900108,Abaíra,BA
9,4200051,Abdon Batista,SC


In [28]:
df_municipio.columns.tolist()

['id_municipio', 'Municipio', 'UF']

In [40]:
df_municipio.describe(include="all")

,id_municipio,Municipio,UF
count,5548,5548,5548
unique,5548,5279,27
top,5200050,Bom Jesus,MG
freq,1,5,853


In [39]:
df_municipio.isna().sum()

id_municipio    1
Municipio       1
UF              1
dtype: int64

In [40]:
df_municipio = df_municipio.dropna(subset=["UF"])

# Salvar novamente no parquet
df_municipio.to_parquet("../data/gold/dimensions/dim_municipio/dim_municipio.parquet", index=False)

In [41]:
df_municipio.isna().sum()

id_municipio    0
Municipio       0
UF              0
dtype: int64

In [42]:
df_municipio.duplicated().sum()

np.int64(0)

In [43]:
df_municipio["UF"].value_counts()

UF
MG    853
SP    642
RS    493
BA    417
PR    399
SC    295
GO    245
PI    224
PB    223
MA    217
PE    185
CE    184
RN    167
PA    144
MT    141
TO    139
AL    102
RJ     92
MS     79
ES     78
SE     75
AM     61
RO     53
AC     22
AP     16
DF      1
RR      1
Name: count, dtype: int64

Ao todo o Brasil possui 5.569 municípios, mas se imagina que nem todos possuem uma meta definida, justificando a divergência para os 5.548 municípios da base GOLD, mas resgatando os dados da camada BRONZE foi possível verificar que os dados são mantidos, não havendo nenhum ponto de atenção aqui.

Havia um municipio nulo que foi devidamente excluído

### Dimensão Estado

In [23]:
import pandas as pd

df_estado = pd.read_parquet("../data/gold/dimensions/dim_uf/dim_uf.parquet")

df_estado.shape

(28, 1)

In [4]:
df_estado.head(5)

,UF
0,AC
1,AL
2,AM
3,AP
4,BA


In [5]:
df_estado["UF"].value_counts()

UF
AC    1
AL    1
AM    1
AP    1
BA    1
CE    1
DF    1
ES    1
GO    1
MA    1
MG    1
MS    1
MT    1
PA    1
PB    1
PE    1
PI    1
PR    1
RJ    1
RN    1
RO    1
RR    1
RS    1
SC    1
SE    1
SP    1
TO    1
Name: count, dtype: int64

In [6]:
df_estado.describe(include="all")

,UF
count,27
unique,27
top,AC
freq,1


In [24]:
df_estado.isna().sum()

UF    1
dtype: int64

In [25]:
df_estado[df_estado["UF"].isna()]

,UF
27,NaN


In [26]:
# Remover linhas onde UF é nulo
df_estado = df_estado.dropna(subset=["UF"])

# Salvar novamente no parquet
df_estado.to_parquet("../data/gold/dimensions/dim_uf/dim_uf.parquet", index=False)

In [27]:
df_estado.isna().sum()

UF    0
dtype: int64

In [12]:
df_estado.duplicated().sum()

np.int64(0)

Dimensão estado possuia um dado nulo que foi descartado, pois não fazia sentido mantê-lo e não havia necessidade de um tratamento tão específico.

### Dimensão rede

In [28]:
import pandas as pd

df_rede= pd.read_parquet("../data/gold/dimensions/dim_rede/dim_rede.parquet")

df_rede.shape

(4, 2)

In [29]:
df_rede.head()

,ID_Rede,Rede
0,2,Estadual
1,3,Municipal
2,4,Privada
3,<NA>,NaN


In [30]:
df_rede = df_rede.dropna(subset=["Rede"])

df_rede.to_parquet("../data/gold/dimensions/dim_rede/dim_rede.parquet", index=False)

In [31]:
df_rede.head()

,ID_Rede,Rede
0,2,Estadual
1,3,Municipal
2,4,Privada


Tabela muito pequena, logo não há necessidade de exploração mais profunda, havia dado nulo que foi devidamente removido

### Dimensão Tempo

In [35]:
import pandas as pd

df_tempo = pd.read_parquet("../data/gold/dimensions/dim_tempo/dim_tempo.parquet")

df_tempo.shape

(2, 1)

In [36]:
df_tempo.head()

,ano
0,2023
1,2024


Sem necessidade de exploração ou limpeza

## Tabelas Fato

In [44]:
import pandas as pd

df_fact_br = pd.read_parquet("../data/gold/facts/fato_alfabetizacao_brasil/fato_alfabetizacao_brasil.parquet")

df_fact_br.shape

(5, 10)

In [46]:
df_fact_br.head()

,ano,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
0,2023,2,155140,80662,131526,131522,51.99,84.78,84.78,NaN
1,2023,3,1592299,796765,1371532,1371287,50.04,86.14,86.12,NaN
2,2024,2,280258,150760,241173,241074,53.79,86.05,86.02,NaN
3,2024,3,1840277,956343,1611591,1610754,51.97,87.57,87.53,NaN
4,2024,4,25,16,24,24,64.00,96.00,96.00,NaN


In [47]:
import pandas as pd

df_fact_uf = pd.read_parquet("../data/gold/facts/fato_alfabetizacao_uf/fato_alfabetizacao_uf.parquet")

df_fact_uf.shape

(100, 11)

In [48]:
df_fact_uf.head()

,ano,UF,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
0,2023,AC,2,61,42,53,53,68.85,86.89,86.89,NaN
1,2023,AC,3,239,92,181,181,38.49,75.73,75.73,NaN
2,2023,AL,2,1620,534,1434,1434,32.96,88.52,88.52,NaN
3,2023,AL,3,33756,13979,31231,31231,41.41,92.52,92.52,NaN
4,2023,AM,2,14041,7461,11810,11810,53.14,84.11,84.11,NaN


In [49]:
df_fact_uf.tail()

,ano,UF,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
95,2024,SE,3,17276,6033,16006,15819,34.92,92.65,91.57,NaN
96,2024,SP,2,105973,63911,96546,96540,60.31,91.10,91.10,NaN
97,2024,SP,3,337815,167676,298945,298913,49.64,88.49,88.48,NaN
98,2024,TO,3,19688,8431,16764,16764,42.82,85.15,85.15,NaN
99,2024,TO,4,25,16,24,24,64.00,96.00,96.00,NaN


In [50]:
df_fact_uf.describe(include="all")

,ano,UF,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
count,100.0,100,100.0,100.00000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,0.0
unique,<NA>,27,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,<NA>,AC,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,<NA>,4,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2023.52,NaN,2.54,38679.99000,19845.460000,33558.460000,33546.610000,50.336600,86.658100,86.606800,NaN
std,0.502117,NaN,0.520683,53388.45883,28553.992855,46672.150966,46666.849805,13.093206,6.624368,6.588981,NaN
min,2023.0,NaN,2.0,25.00000,16.000000,24.000000,24.000000,29.140000,61.360000,61.360000,NaN
25%,2023.0,NaN,2.0,1562.75000,625.000000,1376.250000,1376.250000,39.435000,83.052500,83.052500,NaN
50%,2024.0,NaN,3.0,19864.00000,7513.500000,16441.500000,16441.500000,48.740000,87.985000,87.900000,NaN
75%,2024.0,NaN,3.0,49723.50000,28506.250000,43906.500000,43906.500000,58.547500,90.525000,90.525000,NaN
